# 🫀 ECG Arrhythmia Detection: Multi-Model Benchmark & Analysis

This notebook provides a comprehensive evaluation and visual comparison of 6 deep learning architectures for **cardiac arrhythmia classification** from single-channel 1D ECG signals.

### Benchmarked Architectures:
1. **Baseline 1D-CNN**: 3-layer convolutional reference baseline.
2. **Enhanced 1D-CNN**: Modern 4-stage 1D-CNN with `BatchNorm1d`, `Dropout`, and Global Pooling.
3. **1D-ResNet (ResNet18-1D)**: Deep residual network with skip connections.
4. **1D-BiLSTM**: Bidirectional LSTM with convolutional tokenizer and Attention Pooling.
5. **1D-Transformer**: Temporal Transformer with Multi-Head Self-Attention.
6. **Pretrained ResNet-18 (CWT)**: Transfer learning using 2D Time-Frequency Spectrogram representations.


In [1]:
import os
import sys
# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from src.models import build_model, list_models
from src.data import get_dataloaders
from src.engine import evaluate_model
from src.utils import format_metrics_table, profile_model_efficiency

sns.set_theme(style='whitegrid')
print('Libraries and modules imported successfully!')


ModuleNotFoundError: No module named 'torch'

## 1. Dataset Exploration & ECG Waveform Visualization

In [ ]:
data_dir = '../data'
train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders(data_dir, batch_size=64)

print(f'Train Samples: {len(train_ds):,}')
print(f'Validation Samples: {len(val_ds):,}')
print(f'Test Samples: {len(test_ds):,}')

# Plot sample heartbeats: Normal Sinus Rhythm vs Arrhythmia
normal_samples = []
arrhythmia_samples = []

for x, y in test_loader:
    for sample, label in zip(x, y):
        if label.item() == 0 and len(normal_samples) < 3:
            normal_samples.append(sample.squeeze().numpy())
        elif label.item() == 1 and len(arrhythmia_samples) < 3:
            arrhythmia_samples.append(sample.squeeze().numpy())
    if len(normal_samples) >= 3 and len(arrhythmia_samples) >= 3:
        break

fig, axes = plt.subplots(2, 3, figsize=(15, 6), dpi=150)
t = np.arange(187)

for i in range(3):
    axes[0, i].plot(t, normal_samples[i], color='teal', lw=2)
    axes[0, i].set_title(f'Normal Heartbeat Sample #{i+1}')
    axes[0, i].set_xlabel('Time Step (Sampling Points)')
    axes[0, i].set_ylabel('Normalized Amplitude')
    
    axes[1, i].plot(t, arrhythmia_samples[i], color='crimson', lw=2)
    axes[1, i].set_title(f'Arrhythmia Sample #{i+1}')
    axes[1, i].set_xlabel('Time Step (Sampling Points)')
    axes[1, i].set_ylabel('Normalized Amplitude')

plt.tight_layout()
plt.show()


## 2. Multi-Model Benchmark Leaderboard

Let's load the benchmark results generated across the 1,456 test samples.

In [ ]:
results_csv = '../results/benchmark_summary.csv'
if os.path.exists(results_csv):
    df_results = pd.read_csv(results_csv)
    display(df_results)
else:
    print('Benchmark summary CSV not found. Run `python benchmark.py` to generate.')


## 3. Visual Comparisons: ROC Curves & Precision-Recall Curves

In [ ]:
if os.path.exists('../results/plots/roc_curves.png'):
    display(Image(filename='../results/plots/roc_curves.png'))
if os.path.exists('../results/plots/pr_curves.png'):
    display(Image(filename='../results/plots/pr_curves.png'))


## 4. Confusion Matrices (Clinical Error Distributions)

In [ ]:
if os.path.exists('../results/plots/confusion_matrices.png'):
    display(Image(filename='../results/plots/confusion_matrices.png'))


## 5. Efficiency vs Performance Trade-Off Analysis

In [ ]:
if os.path.exists('../results/plots/efficiency_tradeoff.png'):
    display(Image(filename='../results/plots/efficiency_tradeoff.png')) 